# tensor-unbind — worked example 3: Unbind 3-D pose tensor into named xyz components

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-unbind`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a tensor's last axis represents a fixed-arity set of named components (like x, y, z coordinates), `unbind(dim=-1)` combined with tuple unpacking gives each component a clear name. This avoids magic indices like `v[..., 0]` and makes the code self-documenting.

## Worked solution

**Step 1 — Identify the named axis.**
A batch of 3-D poses of shape `(B, 3)` has the coordinate axis last. The three values at position 0, 1, 2 along that axis are x, y, z.

**Step 2 — Unbind and destructure.**
`x, y, z = poses.unbind(dim=-1)` gives three `(B,)` tensors named clearly.

**Step 3 — Compute using names.**
Now `x + y + z` or `torch.stack([x, -y, z], dim=-1)` reads naturally. Indices would be error-prone.

**Step 4 — Restacking.**
After transformation, `torch.stack([x_new, y_new, z_new], dim=-1)` reassembles the modified components back to `(B, 3)` — the inverse of unbind+destructure.

In [ ]:
import torch as t

t.manual_seed(1)
B = 5
poses = t.randn(B, 3)    # (B, 3) batch of 3-D positions

# Destructure into named components
px, py, pz = poses.unbind(dim=-1)
print('poses shape:', poses.shape)  # (5, 3)
print('px shape:', px.shape)        # (5,)
print('py shape:', py.shape)        # (5,)

# Example: compute the XZ-plane distance (ignoring y)
xz_dist = (px ** 2 + pz ** 2).sqrt()
print('XZ distances:', xz_dist)

# Example: flip y axis (mirror reflection)
py_flipped = -py
reflected = t.stack([px, py_flipped, pz], dim=-1)
print('reflected shape:', reflected.shape)   # (5, 3)

# Verify: components match direct indexing
assert t.allclose(px, poses[:, 0])
assert t.allclose(py, poses[:, 1])
assert t.allclose(pz, poses[:, 2])
print('All components match direct indexing: True')